# Ingest — Viva Insights consumption export

Reads the Consumption Dashboard export from `Files/landing/viva/` and merges it into two Delta
tables: `viva_credits_weekly` and `viva_spending_policy`. Org attributes travel on the metrics
table automatically, including `organisation` separately from `department`; no org file is needed.
Run `Ingest_Org` to materialise `org_attributes`, optionally enriched by manual overrides.

**Why this notebook exists.** The Viva export only reaches back 6 months. Run
this weekly and the Delta table accumulates history indefinitely; skip it for six months and that
history is gone for good.

**Idempotent.** Merges on the natural key, so re-running last week's file updates rather than
duplicates. Safe to re-run, safe to backfill out of order.

**Handles both export shapes.** The identified export carries `UserPrincipalName` + `EntraId`;
the de-identified one carries `PersonId` + `PeopleHistoricalId`. Either is normalised to the same
contract, with `person_id` holding the real UPN when identified and the hashed id when not.

> **You may not need this notebook.** Viva Insights ships a Dataflow Gen2 connector that writes query results straight into a Lakehouse on a schedule - see [Automating the Viva load](../README.md#automating-the-viva-load). Use this notebook when you are working from downloaded CSVs, or when Dataflow Gen2 capacity cost is not worth it.
## Optional Cowork task categories

`CoworkWeeklyMetrics.csv` can be supplied alone or alongside legacy consumption.
Place each export and its `PeopleMetaData.csv` companion in a separate subfolder
under `Files/landing/viva/`. The notebook reads the landing tree but skips `_processed`.
Task rows upsert separately into `cowork_tasks_weekly` at person/week/category/type grain.
Historical metadata is joined within each export; conflicting duplicate keys fail.
No task file means existing task history is left unchanged. Empty task exports do not erase history.
Missing optional credits remain null; malformed numbers, dates or populated duplicate keys fail.
Task counts are reported activity, not successful completions. No value assumptions are applied during ingestion.


In [ ]:
LANDING = "Files/landing/viva"
TBL_METRICS = "viva_credits_weekly"
TBL_POLICY = "viva_spending_policy"

# Set True to archive each file to landing/viva/_processed after a successful merge.
ARCHIVE_AFTER_LOAD = False

In [ ]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable
import notebookutils
import re


def norm(name):
    """Compare column names ignoring case, spaces, dashes and underscores.

    Export headers drift - 'Session count' vs 'SessionCount' vs 'session_count'.
    Matching on a normalised key means a cosmetic rename upstream does not break
    the pipeline silently.
    """
    return re.sub(r"[ _\-]", "", name).lower()


def pick(df, *aliases):
    """Return the actual column name matching any alias, or None."""
    lookup = {norm(c): c for c in df.columns}
    for a in aliases:
        if norm(a) in lookup:
            return lookup[norm(a)]
    return None


def col_or_null(df, name, cast="string"):
    """The column if present, otherwise a typed null - so downstream can
    reference both export shapes' columns unconditionally."""
    return F.col('`' + name.replace('`', '``') + '`').cast(cast) if name else F.lit(None).cast(cast)


PERSON_ALIASES = {
    'user_principal_name': ['UserPrincipalName', 'UPN', 'UserPrincipal', 'Email', 'Mail', 'EmailAddress'],
    'person_id': ['PersonId'],
    'display_name': ['DisplayName', 'Name', 'FullName', 'PreferredName'],
    'department': ['Department', 'Dept'],
    'organisation': ['Organisation', 'Organization'],
    'job_title': ['JobTitle', 'Title', 'Role'],
    'job_family': ['JobFamily', 'Function', 'FunctionType', 'JobFunction'],
    'city': ['City', 'OfficeLocation', 'Location'],
    'country': ['Country', 'CountryOrRegion', 'Region'],
    'cost_center': ['CostCenter', 'CostCentre'],
    'manager': ['Manager', 'ManagerName', 'Supervisor', 'ManagerId', 'ManagerUPN'],
    'business_unit': ['BusinessUnit', 'Division', 'Segment'],
}
ORG_COLUMNS = [c for c in PERSON_ALIASES if c not in ('person_id', 'user_principal_name')]


def normalise_person(record):
    """One identity contract for CSV facts and automatic/manual org rows."""
    lookup = {}
    for name, value in record.items():
        text = str(value).strip() if value is not None else ''
        if text:
            lookup.setdefault(norm(name), text)
    result = {}
    for canon, aliases in PERSON_ALIASES.items():
        result[canon] = next((lookup[norm(a)] for a in [canon] + aliases if norm(a) in lookup), None)
    for key in ('person_id', 'user_principal_name'):
        if result[key]:
            result[key] = result[key].lower()
    result['person_id'] = result['user_principal_name'] or result['person_id']
    return result


def identity_sql(column):
    return f"NULLIF(LOWER(REGEXP_REPLACE({column}, r'(?U)^[\\s\\x1c-\\x1f]+|[\\s\\x1c-\\x1f]+$', '')), '')"


def add_person_columns(df):
    schema = ', '.join(f'{c} string' for c in PERSON_ALIASES)
    project = F.udf(lambda row: normalise_person(row.asDict()), schema)
    return df.withColumn('_person', project(F.struct(*[col_or_null(df, c).alias(c) for c in df.columns])))

# BEGIN COWORK TASK HELPERS
"""Task ingestion helpers embedded in the Fabric Viva notebook (no client-side writes)."""
import datetime
import decimal
import math
import re

TASK_COLUMNS = {
    "person_id": ("PersonId",),
    "user_principal_name": ("UserPrincipalName", "UPN"),
    "metric_date": ("MetricDate",),
    "task_category": ("TaskCategory",),
    "task_type": ("TaskType",),
    "credits_used": ("CreditsUsed", "Total credits used"),
    "task_count": ("TaskCount", "Total Cowork tasks"),
    "people_historical_id": ("PeopleHistoricalId",),
    "organization": ("Organization", "Organisation"),
    "function_type": ("FunctionType",),
    "is_copilot_licensed": ("IsCopilotLicensed",),
}
TASK_SCHEMA = (
    "person_id string, user_principal_name string, metric_date date, "
    "task_category string, task_type string, credits_used double, task_count long, "
    "people_historical_id string, organization string, function_type string, is_copilot_licensed boolean"
)
TASK_KEYS = ["person_id", "metric_date", "category_key", "type_key"]


def task_key(name):
    return re.sub("[^a-z0-9]", "", name.lower())


def task_error(field, problem):
    raise ValueError(f"Cowork tasks: {field}: {problem}")


def task_bind(record, columns):
    result = {}
    for canonical, aliases in columns.items():
        # Match the M contract's exact PascalCase canonical spelling first.
        preferred = aliases[0]
        matches = [name for name in record if task_key(name) in {
            task_key(alias) for alias in (canonical,) + aliases}]
        if preferred in matches:
            name = preferred
        elif len(matches) > 1:
            task_error(canonical, "ambiguous aliases; keep one source column")
        else:
            name = matches[0] if matches else None
        value = record.get(name) if name else None
        result[canonical] = str(value).strip() if value is not None else None
        if result[canonical] == "":
            result[canonical] = None
    return result


def task_number(value, field, optional=False, integral=False):
    if value is None:
        if optional:
            return None
        task_error(field, "a required numeric value is missing")
    try:
        parsed = decimal.Decimal(value)
    except decimal.InvalidOperation:
        task_error(field, "expected a finite nonnegative number")
    if not parsed.is_finite() or parsed < 0:
        task_error(field, "expected a finite nonnegative number")
    if integral:
        if parsed != parsed.to_integral_value() or parsed > 9007199254740991:
            task_error(field, "expected a nonnegative whole count within the supported integer range")
        return int(parsed)
    value = float(parsed)
    if not math.isfinite(value):
        task_error(field, "expected a finite nonnegative number")
    return value


def task_licence(value):
    if value is None:
        return None
    if value.lower() in ("true", "yes", "1"):
        return True
    if value.lower() in ("false", "no", "0"):
        return False
    task_error("is_copilot_licensed", "expected true/false, yes/no or 1/0")


def normalise_task(record):
    row = task_bind(record, TASK_COLUMNS)
    for key in ("person_id", "user_principal_name", "people_historical_id"):
        row[key] = row[key].lower() if row[key] else None
    row["person_id"] = row["person_id"] or row["user_principal_name"]
    if not row["person_id"]:
        task_error("person_id", "a populated PersonId or UserPrincipalName is required")
    date = row["metric_date"]
    if not date or not re.fullmatch(r"\d{4}-\d{2}-\d{2}", date):
        task_error("metric_date", "expected an ISO week-start date (yyyy-MM-dd)")
    try:
        row["metric_date"] = datetime.date.fromisoformat(date)
    except ValueError:
        task_error("metric_date", "expected a valid ISO week-start date")
    row["task_count"] = task_number(row["task_count"], "task_count", integral=True)
    row["credits_used"] = task_number(row["credits_used"], "credits_used", optional=True)
    row["is_copilot_licensed"] = task_licence(row["is_copilot_licensed"])
    for key in ("task_category", "task_type"):
        row[key] = row[key] or "Not supplied"
    return row


def normalise_task_metadata(record):
    names = ("people_historical_id", "organization", "function_type", "is_copilot_licensed")
    row = task_bind(record, {name: TASK_COLUMNS[name] for name in names})
    row["people_historical_id"] = row["people_historical_id"].lower() if row["people_historical_id"] else None
    row["is_copilot_licensed"] = task_licence(row["is_copilot_licensed"])
    return row


def viva_landing_files(fs, path):
    """Inspect only the configured landing tree, excluding archived/hidden data."""
    files = []
    for entry in fs.ls(path):
        if entry.name.startswith(("_", ".")):
            continue
        if entry.isDir:
            files.extend(viva_landing_files(fs, entry.path))
        elif entry.name.lower().endswith(".csv"):
            files.append(entry.path)
    return sorted(files)


def task_project(frame, function, schema, functions):
    project = functions.udf(lambda row: function(row.asDict()), schema)
    fields = [functions.col("`" + name.replace("`", "``") + "`").alias(name) for name in frame.columns]
    return frame.select(project(functions.struct(*fields)).alias("_task")).select("_task.*")


def task_read_project(spark, path, function, schema, functions):
    # CSV cannot infer even a header schema from a zero-byte file.
    if spark.read.text(path).limit(1).count() == 0:
        return spark.createDataFrame([], schema)
    raw = spark.read.option("header", True).option("inferSchema", False).csv(path)
    return task_project(raw, function, schema, functions)


def ingest_tasks(spark, paths, landing_files, table_name, functions, delta):
    """Separate optional Delta fact; never union task activity into legacy billing."""
    if not paths:
        print("Task categories not supplied; existing task history is unchanged.")
        return []
    combined = None
    consumed = []
    for path in paths:
        tasks = task_read_project(spark, path, normalise_task, TASK_SCHEMA, functions)
        companion = next((p for p in landing_files if p.rsplit("/", 1)[0] == path.rsplit("/", 1)[0]
                          and p.rsplit("/", 1)[-1].lower() == "peoplemetadata.csv"), None)
        consumed.append(path)
        if companion:
            metadata = task_read_project(spark, companion, normalise_task_metadata,
                "people_historical_id string, organization string, function_type string, is_copilot_licensed boolean",
                functions).filter(functions.col("people_historical_id").isNotNull()).dropDuplicates()
            if metadata.groupBy("people_historical_id").count().filter(functions.col("count") > 1).limit(1).count():
                task_error("people_historical_id", "conflicting metadata rows; reconcile the companion export")
            tasks = tasks.alias("t").join(metadata.alias("m"), "people_historical_id", "left").select(
                *[functions.coalesce(functions.col("t." + name), functions.col("m." + name)).alias(name)
                  if name in ("organization", "function_type", "is_copilot_licensed")
                  else functions.col(name if name == "people_historical_id" else "t." + name).alias(name)
                  for name in TASK_COLUMNS])
            consumed.append(companion)
        combined = tasks if combined is None else combined.unionByName(tasks)
    combined = (combined.withColumn("category_key", functions.lower(functions.col("task_category")))
                .withColumn("type_key", functions.lower(functions.col("task_type"))).dropDuplicates())
    if combined.groupBy(*TASK_KEYS).count().filter(functions.col("count") > 1).limit(1).count():
        task_error("task_count", "conflicting rows share person, week, category and type")
    combined = combined.withColumn("_loaded_at", functions.current_timestamp())
    if spark.catalog.tableExists(table_name):
        existing = spark.table(table_name)
        if existing.groupBy(*TASK_KEYS).count().filter(functions.col("count") > 1).limit(1).count():
            task_error("task_count", "existing task history has duplicate keys; reconcile before merging")
        condition = " AND ".join(f"t.{key} = s.{key}" for key in TASK_KEYS)
        (delta.forName(spark, table_name).alias("t").merge(combined.alias("s"), condition)
            .whenMatchedUpdateAll().whenNotMatchedInsertAll().execute())
    else:
        combined.write.format("delta").saveAsTable(table_name)
    print(f"{table_name}: processed {combined.count():,} task rows; historical keys are upserted.")
    return consumed


## Metrics

In [ ]:
# Both export families are optional individually, but an empty landing path is an input error.
landing_files = viva_landing_files(notebookutils.fs, LANDING)
files = [p for p in landing_files if p.rsplit("/", 1)[-1].lower().startswith("personservicecreditsmetrics")]
task_files = [p for p in landing_files if task_key(p.rsplit("/", 1)[-1]).startswith("coworkweeklymetrics")]
policy_files = [p for p in landing_files if p.rsplit("/", 1)[-1].lower().startswith("spendingpolicymetadata")]
if not files and not task_files:
    raise ValueError("No supported consumption or task CSV files found in the configured landing folder.")
raw = None
for path in files:
    frame = spark.read.option("header", True).option("inferSchema", False).csv(path)
    raw = frame if raw is None else raw.unionByName(frame, allowMissingColumns=True)

if raw is not None:
    print(f"read {raw.count():,} consumption rows")
    c_upn = pick(raw, *PERSON_ALIASES["user_principal_name"])
    c_pid = pick(raw, "PersonId")
    c_phid = pick(raw, "PeopleHistoricalId")
    c_entra = pick(raw, "EntraId", "ObjectId", "AadObjectId")
    if not (c_upn or c_pid):
        raise ValueError("Consumption requires UserPrincipalName or PersonId.")
else:
    print("No legacy consumption file supplied; legacy tables are unchanged.")


In [ ]:
if files:
    # person_id is the join key everywhere downstream: the real UPN when the export
    # is identified, the hashed id when it is not.
    metrics = add_person_columns(raw).select(
        F.col('_person.person_id').alias('person_id'),
        F.col('_person.user_principal_name').alias('user_principal_name'),
        col_or_null(raw, c_entra).alias("entra_id"),
        col_or_null(raw, c_phid).alias("people_historical_id"),
        F.col(pick(raw, "ServiceId")).cast("string").alias("service_id"),
        F.col(pick(raw, "ServiceName")).cast("string").alias("service_name"),
        F.col(pick(raw, "SpendingPolicyId")).cast("string").alias("spending_policy_id"),
        F.to_date(F.col(pick(raw, "MetricDate"))).alias("metric_date"),
        F.col(pick(raw, "Session count")).cast("int").alias("session_count"),
        F.col(pick(raw, "Spending policy limit")).cast("long").alias("spending_policy_limit"),
        F.col(pick(raw, "Total Copilot Credits used")).cast("double").alias("credits_used"),
        F.col(pick(raw, "User limit")).cast("long").alias("user_limit"),
        *[F.col(f'_person.{c}').alias(c) for c in ORG_COLUMNS],
    ).withColumn("_loaded_at", F.current_timestamp())
    if metrics.filter(F.col('person_id').isNull()).limit(1).count():
        raise ValueError('Consumption contains a row with no populated UPN alias or PersonId.')

    weeks = metrics.select("metric_date").distinct().count()
    print(f"{metrics.count():,} rows across {weeks} weeks, "
          f"{metrics.select('person_id').distinct().count():,} people")
    metrics.select(F.min("metric_date").alias("from"),
                   F.max("metric_date").alias("to")).show()

In [ ]:
if files:
    # A person can hold rows under more than one policy in the same week - usage
    # outside any policy window is recorded against the all-zero GUID. The policy is
    # therefore part of the key, or a merge would collapse two legitimate rows into
    # one and lose credits.
    KEY = ["person_id", "service_id", "spending_policy_id", "metric_date"]
    metrics = metrics.dropDuplicates([c for c in metrics.columns if c != '_loaded_at'])
    if metrics.groupBy(*KEY).count().filter(F.col('count') > 1).limit(1).count():
        raise ValueError('Conflicting consumption rows share a normalised merge key; reconcile the input files.')

    if spark.catalog.tableExists(TBL_METRICS):
        existing = set(spark.table(TBL_METRICS).columns)
        added = [c for c in ORG_COLUMNS if c not in existing]
        if added:
            spark.sql(f"ALTER TABLE {TBL_METRICS} ADD COLUMNS (" + ', '.join(f'{c} STRING' for c in added) + ')')
        before = spark.table(TBL_METRICS).count()
        tgt = DeltaTable.forName(spark, TBL_METRICS)
        # Also match pre-fix rows whose stored identity was not normalised.
        old_key = 'COALESCE(' + identity_sql('t.user_principal_name') + ', ' + identity_sql('t.person_id') + ')'
        if (spark.table(TBL_METRICS).alias('t').groupBy(F.expr(old_key), *KEY[1:]).count()
                .filter(F.col('count') > 1).limit(1).count()):
            raise ValueError('Existing consumption has colliding normalised keys; reconcile duplicates before merging.')
        cond = f"{old_key} = s.person_id AND " + " AND ".join(f"t.{k} <=> s.{k}" for k in KEY[1:])
        updates = {c: f's.{c}' for c in metrics.columns}
        updates.update({c: f"COALESCE(NULLIF(TRIM(s.{c}), ''), t.{c})" for c in ORG_COLUMNS})
        (tgt.alias("t")
            .merge(metrics.alias("s"), cond)
            .whenMatchedUpdate(set=updates)
            .whenNotMatchedInsertAll()
            .execute())
        after = spark.table(TBL_METRICS).count()
        print(f"{TBL_METRICS}: {before:,} -> {after:,}  (+{after - before:,} new)")
    else:
        metrics.write.format("delta").saveAsTable(TBL_METRICS)
        print(f"{TBL_METRICS}: created with {metrics.count():,} rows")

## Spending policies

Small and slow-moving, so this one is a full replace rather than a merge.

In [ ]:
if policy_files:
    pol_raw = None
    for path in policy_files:
        frame = spark.read.option("header", True).csv(path)
        pol_raw = frame if pol_raw is None else pol_raw.unionByName(frame, allowMissingColumns=True)

    policy = pol_raw.select(
        F.col(pick(pol_raw, "SpendingPolicyId")).cast("string").alias("spending_policy_id"),
        # The all-zero GUID row has no name - it marks usage outside any policy.
        F.when(F.col(pick(pol_raw, "Name")).isNull() | (F.col(pick(pol_raw, "Name")) == ""),
               F.lit("(Unassigned)"))
         .otherwise(F.col(pick(pol_raw, "Name"))).alias("name"),
        F.coalesce(F.col(pick(pol_raw, "PlanLimit")).cast("long"), F.lit(0)).alias("plan_limit"),
        F.coalesce(F.col(pick(pol_raw, "UserLimit")).cast("long"), F.lit(0)).alias("user_limit"),
        F.col(pick(pol_raw, "IncludedServices")).cast("string").alias("included_services"),
    ).withColumn("_loaded_at", F.current_timestamp())

    policy.write.format("delta").mode("overwrite") \
        .option("overwriteSchema", "true").saveAsTable(TBL_POLICY)
    print(f"{TBL_POLICY}: {policy.count()} policies")
    policy.show(truncate=False)

## Check

Accumulated history is the whole point of this path, so it is worth looking at every run.

In [ ]:
if spark.catalog.tableExists(TBL_METRICS):
    spark.sql(f"""
        SELECT  MIN(metric_date)                    AS earliest_week,
                MAX(metric_date)                    AS latest_week,
                COUNT(DISTINCT metric_date)         AS weeks_held,
                COUNT(DISTINCT person_id)           AS people,
                ROUND(SUM(credits_used))            AS total_credits
        FROM    {TBL_METRICS}
    """).show(truncate=False)

    # Weeks_held above the export's own 6-month window means history is accruing -
    # which is the reason to be on this path at all.
    spark.sql(f"""
        SELECT  metric_date, COUNT(*) AS rows, ROUND(SUM(credits_used)) AS credits
        FROM    {TBL_METRICS}
        GROUP BY metric_date
        ORDER BY metric_date DESC
        LIMIT   12
    """).show(truncate=False)

# Optional task ingestion remains separate from legacy credit billing.
task_consumed = ingest_tasks(spark, task_files, landing_files, "cowork_tasks_weekly", F, DeltaTable)


In [ ]:
if ARCHIVE_AFTER_LOAD:
    stamp = datetime.datetime.now(datetime.timezone.utc).strftime("%Y%m%d_%H%M%S")
    dest = f"{LANDING}/_processed/{stamp}"
    for path in sorted(set(files + policy_files + task_consumed)):
        relative = path.split(LANDING.rstrip("/") + "/", 1)[-1]
        target = f"{dest}/{relative}"
        notebookutils.fs.mkdirs(target.rsplit("/", 1)[0])
        notebookutils.fs.mv(path, target)
    print("Archived successfully ingested files, retaining their export subfolders.")
else:
    print("ARCHIVE_AFTER_LOAD is False - files left in place.")
    print("Re-running upserts the same natural keys without adding task credits to billing.")
